# 07b. Avaliar predictions `.gz` — atribuição mãe + endereço

Lê todos `*.gz` de um diretório (não recursivo), deduplica pares,
aplica a escada do [`04_atribuir_mae_endereco.ipynb`](04_atribuir_mae_endereco.ipynb)
e métricas no espírito do [`05_avaliar_mae_endereco.ipynb`](05_avaliar_mae_endereco.ipynb)
/ [`06_numeros_finais.ipynb`](06_numeros_finais.ipynb).

Formato por linha (whitespace):

`status  X  id1  id2  total_score  match_probability  s1 … sN`

- colunas 0–5 fixas; resto = parciais (não entram na escada)
- pares repetidos entre regras → fica o maior `match_probability`
  (empate: maior `total_score`)

Ajuste `PRED_GZ_DIR` e `PREFIXAR_IDS` na célula de config.


In [ ]:
import csv
import gzip
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    LISTA_OURO_ARQUIVO,
    MAX_CENSOS_POR_CPF,
    OUTPUT_DIR,
    SPLINK_INPUT_VIEW,
    TABELA_CENSO_LIMPA,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    cpf_norm_sql,
    drop_splink_temp_tables,
    ensure_output_dir,
    export_parquet,
    get_connection,
    lista_ouro_colunas,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

# --- config ---
PRED_GZ_DIR = Path('/data/spark/shared/datasets/predictions_gz')  # ajuste
# Se id1/id2 vierem sem prefixo, monta censo_{id1} / cpf_{id2}
PREFIXAR_IDS = True

PISO_MODELO = 0.90
PISO_RESGATE = 0.50
PISO_BAIXO = 0.25
TETO = MAX_CENSOS_POR_CPF

ATRIB_OUT = OUTPUT_DIR / 'splink_atribuicao_mae_endereco_gz.parquet'
STAGING_CSV = OUTPUT_DIR / 'pred_gz_staging.csv'

print_paths()
ensure_output_dir()
print('PRED_GZ_DIR:', PRED_GZ_DIR)
print('PREFIXAR_IDS:', PREFIXAR_IDS)
print('Atribuição (saída):', ATRIB_OUT)

if not PRED_GZ_DIR.is_dir():
    raise FileNotFoundError(f'Diretório não encontrado: {PRED_GZ_DIR}')

gz_files = sorted(PRED_GZ_DIR.glob('*.gz'))
if not gz_files:
    raise FileNotFoundError(f'Nenhum *.gz em {PRED_GZ_DIR}')
print(f'Arquivos .gz: {len(gz_files)}')

con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO, TABELA_CENSO_LIMPA],
    notebook_origem='00b',
)
require_input(LISTA_OURO_ARQUIVO, label='LISTA_OURO')
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print(
    f'Faixas: modelo>={PISO_MODELO} | resgate>={PISO_RESGATE} | '
    f'baixo>={PISO_BAIXO} | teto={TETO}'
)


## Carga dos `.gz`

Parse com `split()`; grava staging CSV (6 colunas + arquivo) e lê no DuckDB.


In [ ]:
n_linhas = 0
n_curtas = 0

with STAGING_CSV.open('w', newline='') as out:
    w = csv.writer(out)
    w.writerow([
        'status', 'x', 'id1', 'id2',
        'total_score', 'match_probability', 'arquivo',
    ])
    for path in gz_files:
        with gzip.open(path, 'rt') as f:
            for line in f:
                parts = line.split()
                if len(parts) < 6:
                    n_curtas += 1
                    continue
                w.writerow([
                    parts[0], parts[1], parts[2], parts[3],
                    parts[4], parts[5], path.name,
                ])
                n_linhas += 1

print(f'Linhas ok: {n_linhas:,} | curtas (<6 cols): {n_curtas:,}')
print('Staging:', STAGING_CSV)

con.execute(f'''
CREATE OR REPLACE TABLE pred_raw AS
SELECT
    CAST(status AS VARCHAR) AS status,
    CAST(x AS VARCHAR) AS x,
    CAST(id1 AS VARCHAR) AS id1,
    CAST(id2 AS VARCHAR) AS id2,
    TRY_CAST(total_score AS DOUBLE) AS total_score,
    TRY_CAST(match_probability AS DOUBLE) AS match_probability,
    CAST(arquivo AS VARCHAR) AS arquivo
FROM read_csv(
    '{STAGING_CSV}',
    header=true,
    auto_detect=true
)
''')

display(con.execute('''
SELECT * FROM pred_raw LIMIT 5
''').df())
display(con.execute('''
SELECT
    COUNT(*) AS n,
    COUNT(DISTINCT id1 || '|' || id2) AS n_pares_distintos,
    COUNT(DISTINCT arquivo) AS n_arquivos
FROM pred_raw
''').df())


## Dedupe e IDs Splink

Por `(id1, id2)`: maior `match_probability`, empate maior `total_score`.


In [ ]:
if PREFIXAR_IDS:
    id_censo_sql = """
        CASE
            WHEN id1 LIKE 'censo_%' THEN id1
            ELSE 'censo_' || id1
        END
    """
    id_cpf_sql = """
        CASE
            WHEN id2 LIKE 'cpf_%' THEN id2
            ELSE 'cpf_' || id2
        END
    """
else:
    id_censo_sql = 'id1'
    id_cpf_sql = 'id2'

con.execute(f'''
CREATE OR REPLACE TABLE pred_dedup AS
SELECT
    {id_censo_sql} AS unique_id_censo,
    {id_cpf_sql} AS unique_id_cpf,
    match_probability,
    total_score,
    status,
    x,
    arquivo
FROM pred_raw
WHERE match_probability IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id1, id2
    ORDER BY match_probability DESC, total_score DESC NULLS LAST
) = 1
''')

# View no formato que o 04 espera
_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')

con.execute('''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM pred_dedup
''')

n_raw = con.execute('SELECT COUNT(*) FROM pred_raw').fetchone()[0]
n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]
print(f'Bruto: {n_raw:,} → únicos: {n_pred:,} (removidos {n_raw - n_pred:,})')

# cobertura dos ids no splink_input (join da escada)
display(con.execute(f'''
SELECT
    COUNT(*) AS n_pred,
    COUNT(*) FILTER (
        WHERE ca.unique_id IS NOT NULL AND pb.unique_id IS NOT NULL
    ) AS n_com_join_input,
    COUNT(*) FILTER (WHERE ca.unique_id IS NULL) AS n_censo_fora_input,
    COUNT(*) FILTER (WHERE pb.unique_id IS NULL) AS n_cpf_fora_input
FROM splink_predictions p
LEFT JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
LEFT JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
''').df())

display(con.execute('''
SELECT
    CASE
        WHEN match_probability >= 0.90 THEN 'p_ge_090'
        WHEN match_probability >= 0.50 THEN 'p_050_090'
        WHEN match_probability >= 0.25 THEN 'p_025_050'
        WHEN match_probability >= 0.05 THEN 'p_005_025'
        ELSE 'p_lt_005'
    END AS faixa_p,
    COUNT(*) AS n
FROM splink_predictions
GROUP BY 1
ORDER BY 1 DESC
''').df())


## Pares elegíveis ao resgate (R1 / R2 / R3)

Mesma lógica do 04.


In [ ]:
tok_ca = "string_split(ca.nome_completo_phon, ' ')"
tok_pb = "string_split(pb.nome_completo_phon, ' ')"
n_ca = f"len({tok_ca})"
n_pb = f"len({tok_pb})"
n_min = f"least({n_ca}, {n_pb})"
n_inter = f"len(list_intersect({tok_ca}, {tok_pb}))"
cobertura = f"({n_inter} * 1.0 / nullif({n_min}, 0))"
contido = (
    f"{n_min} >= 2 AND {n_ca} <> {n_pb} AND {n_inter} = {n_min}"
)
primeiro_ok = (
    "ca.primeiro_nome_phon IS NOT NULL AND pb.primeiro_nome_phon IS NOT NULL "
    "AND ("
    "ca.primeiro_nome_phon = pb.primeiro_nome_phon "
    "OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) <= 1"
    ")"
)
data_igual = (
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento = pb.data_nascimento"
)
data_ambas = (
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento <> pb.data_nascimento"
)
delta_dias = (
    "abs(date_diff('day', "
    "CAST(ca.data_nascimento AS DATE), CAST(pb.data_nascimento AS DATE)))"
)
mesmo_dia_mes = (
    "day(CAST(ca.data_nascimento AS DATE)) = day(CAST(pb.data_nascimento AS DATE)) "
    "AND month(CAST(ca.data_nascimento AS DATE)) = month(CAST(pb.data_nascimento AS DATE))"
)
delta_ano = (
    "abs(year(CAST(ca.data_nascimento AS DATE)) "
    "- year(CAST(pb.data_nascimento AS DATE)))"
)
data_plausivel = (
    f"({delta_dias} <= 1) OR "
    f"(({mesmo_dia_mes}) AND ({delta_ano}) = 1) OR "
    f"(({mesmo_dia_mes}) AND ({delta_ano}) = 10)"
)
nome_muito_forte = (
    f"(ca.nome_completo_phon = pb.nome_completo_phon) OR ({contido})"
)
mae_presente = (
    "ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL"
)
jw_mae = "jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon)"

r1_medio = (
    f"({data_igual}) AND ({primeiro_ok}) AND "
    f"(({contido}) OR ({cobertura} >= 0.75))"
)
r1_baixo = (
    f"({data_igual}) AND ({primeiro_ok}) AND "
    f"(({contido}) OR ({cobertura} >= 0.80))"
)
r2 = (
    f"({nome_muito_forte}) AND ({data_ambas}) AND ({data_plausivel})"
)
r3_medio = (
    f"({primeiro_ok}) AND ({cobertura} >= 0.55) AND ({mae_presente}) "
    f"AND ({jw_mae} >= 0.75)"
)

desempate = (
    "("
    "CAST(ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL AS INTEGER) "
    "+ CAST("
    "ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL "
    "AND ca.logradouro_norm = pb.logradouro_norm AS INTEGER) "
    "+ CAST("
    "ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep AS INTEGER)"
    ")"
)

con.execute(f'''
CREATE OR REPLACE TABLE pares AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    {desempate} AS desempate,
    ({r1_medio}) AS r1_medio,
    ({r1_baixo}) AS r1_baixo,
    ({r2}) AS r2,
    ({r3_medio}) AS r3_medio,
    CASE
        WHEN ({r1_medio}) THEN 'R1'
        WHEN ({r2}) THEN 'R2'
        WHEN ({r3_medio}) THEN 'R3'
    END AS regra_medio,
    CASE
        WHEN ({r1_baixo}) THEN 'R1'
        WHEN ({r2}) THEN 'R2'
    END AS regra_baixo
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
WHERE p.match_probability >= {PISO_BAIXO}
''')

n_pares = con.execute('SELECT COUNT(*) FROM pares').fetchone()[0]
print('Pares com p >=', PISO_BAIXO, 'e join no input:', f'{n_pares:,}')
display(con.execute('''
SELECT
    COUNT(*) FILTER (WHERE r1_medio) AS n_r1_medio,
    COUNT(*) FILTER (WHERE r2) AS n_r2,
    COUNT(*) FILTER (WHERE r3_medio) AS n_r3_medio,
    COUNT(*) FILTER (WHERE r1_baixo) AS n_r1_baixo,
    COUNT(*) FILTER (WHERE regra_medio IS NOT NULL) AS n_com_regra_medio,
    COUNT(*) FILTER (WHERE regra_baixo IS NOT NULL) AS n_com_regra_baixo
FROM pares
''').df())


## Atribuição por faixa

Três passos iguais ao 04: modelo → resgate → baixo.


In [ ]:
con.execute('''
CREATE OR REPLACE TABLE atribuicao (
    unique_id_censo VARCHAR,
    unique_id_cpf VARCHAR,
    match_probability DOUBLE,
    faixa VARCHAR,
    regra VARCHAR
)
''')

faixas = [
    ('modelo', PISO_MODELO, None, None),
    ('resgate', PISO_RESGATE, PISO_MODELO, 'regra_medio'),
    ('baixo', PISO_BAIXO, PISO_RESGATE, 'regra_baixo'),
]

for nome_faixa, piso, teto_nota, col_regra in faixas:
    if teto_nota is None:
        filtro_p = f'match_probability >= {piso}'
    else:
        filtro_p = (
            f'match_probability >= {piso} '
            f'AND match_probability < {teto_nota}'
        )
    if col_regra is None:
        filtro_regra = 'TRUE'
        regra_sql = "'score'"
    else:
        filtro_regra = f'{col_regra} IS NOT NULL'
        regra_sql = col_regra

    con.execute(f'''
    CREATE OR REPLACE TABLE faixa_base AS
    SELECT
        unique_id_censo,
        unique_id_cpf,
        match_probability,
        desempate,
        {regra_sql} AS regra
    FROM pares
    WHERE {filtro_p}
      AND {filtro_regra}
      AND unique_id_censo NOT IN (SELECT unique_id_censo FROM atribuicao)
      AND unique_id_cpf NOT IN (
          SELECT unique_id_cpf
          FROM atribuicao
          GROUP BY 1
          HAVING COUNT(*) >= {TETO}
      )
    ''')

    con.execute('''
    CREATE OR REPLACE TABLE faixa_melhor AS
    SELECT unique_id_censo, unique_id_cpf, match_probability, regra
    FROM faixa_base
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY unique_id_censo
        ORDER BY match_probability DESC, desempate DESC, unique_id_cpf
    ) = 1
    ''')

    antes = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
    con.execute(f'''
    INSERT INTO atribuicao
    SELECT
        m.unique_id_censo,
        m.unique_id_cpf,
        m.match_probability,
        '{nome_faixa}' AS faixa,
        m.regra
    FROM faixa_melhor m
    JOIN (
        SELECT unique_id_cpf, COUNT(*) AS n_novo
        FROM faixa_melhor
        GROUP BY 1
    ) g ON g.unique_id_cpf = m.unique_id_cpf
    LEFT JOIN (
        SELECT unique_id_cpf, COUNT(*) AS n_ja
        FROM atribuicao
        GROUP BY 1
    ) j ON j.unique_id_cpf = m.unique_id_cpf
    WHERE coalesce(j.n_ja, 0) + g.n_novo <= {TETO}
    ''')
    depois = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
    print(f'{nome_faixa} ({filtro_p}): +{depois - antes:,} (total {depois:,})')


## Cobertura


In [ ]:
n_app = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''').fetchone()[0]
n_ok = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
print(f'Censo aplicação: {n_app:,}')
print(f'Atribuídos: {n_ok:,} ({100.0 * n_ok / n_app:.1f}%)')

display(con.execute('''
SELECT
    faixa,
    regra,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao
GROUP BY 1, 2
ORDER BY
    CASE faixa
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
    END,
    regra
''').df())

display(con.execute('''
SELECT
    n_censos,
    COUNT(*) AS n_cpf,
    n_censos * COUNT(*) AS n_pares
FROM (
    SELECT unique_id_cpf, COUNT(*) AS n_censos
    FROM atribuicao
    GROUP BY 1
)
GROUP BY 1
ORDER BY 1
''').df())

export_parquet(con, 'atribuicao', path=ATRIB_OUT)
print('Exportado:', ATRIB_OUT)


## Qualidade dos pares atribuídos


In [ ]:
display(con.execute(f'''
SELECT
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
    ) / COUNT(*), 1) AS pct_mae_preenchida,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
          AND ca.nome_mae_phon = pb.nome_mae_phon
    ) / COUNT(*), 1) AS pct_mae_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
          AND ca.logradouro_norm = pb.logradouro_norm
    ) / COUNT(*), 1) AS pct_logradouro_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
    ) / COUNT(*), 1) AS pct_cep_igual,
    ROUND(100.0 * COUNT(*) FILTER (
        WHERE ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
    ) / COUNT(*), 1) AS pct_data_igual
FROM atribuicao a
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = a.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = a.unique_id_cpf
''').df())


## Quem ficou de fora

Melhor par no predict por Censo sem atribuição.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE melhor_qualquer AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY
        p.match_probability DESC,
        CAST(
            ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
            AS INTEGER
        ) DESC,
        CAST(
            ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL
            AND ca.logradouro_norm = pb.logradouro_norm
            AS INTEGER
        ) DESC,
        CAST(
            ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep
            AS INTEGER
        ) DESC,
        p.unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id AS unique_id_censo
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
    SELECT 1 FROM atribuicao a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute('''
CREATE OR REPLACE TABLE sem_cpf_melhor AS
SELECT
    s.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    CASE
        WHEN m.match_probability IS NULL THEN 'sem_par'
        WHEN m.match_probability >= 0.50 THEN 'p_ge_050'
        WHEN m.match_probability >= 0.25 THEN 'p_025_050'
        WHEN m.match_probability >= 0.05 THEN 'p_005_025'
        ELSE 'p_lt_005'
    END AS faixa_p
FROM censo_sem_cpf s
LEFT JOIN melhor_qualquer m ON m.unique_id_censo = s.unique_id_censo
''')

n_sem = con.execute('SELECT COUNT(*) FROM censo_sem_cpf').fetchone()[0]
assert n_ok + n_sem == n_app

display(con.execute(f'''
SELECT
    a.faixa_p,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / nullif({n_sem}, 0), 1) AS pct_sem_cpf,
    ROUND(100.0 * coalesce(g.n, 0) / {n_app}, 1) AS pct_aplicacao
FROM (VALUES
    (1, 'p_ge_050'),
    (2, 'p_025_050'),
    (3, 'p_005_025'),
    (4, 'p_lt_005'),
    (5, 'sem_par')
) a(ord, faixa_p)
LEFT JOIN (
    SELECT faixa_p, COUNT(*) AS n
    FROM sem_cpf_melhor
    GROUP BY 1
) g ON g.faixa_p = a.faixa_p
ORDER BY a.ord
''').df())


## Lista ouro (`censo_limpo`)

Fatia `lista_ouro` / `pareamento` / `sem_cpf` como no 06.
Quem está na ouro e no pareamento conta como `lista_ouro`.
`nos_dois` deve ser 0 se a ouro saiu da aplicação.


In [ ]:
ouro_cols = {
    r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{LISTA_OURO_ARQUIVO}') LIMIT 0"
    ).fetchall()
}
id_ouro, cpf_ouro = lista_ouro_colunas(ouro_cols)

con.execute(f'''
CREATE OR REPLACE TABLE lista_ouro AS
SELECT
    CAST({id_ouro} AS VARCHAR) AS id_censo,
    {cpf_norm_sql(cpf_ouro)} AS cpf_norm
FROM read_parquet('{LISTA_OURO_ARQUIVO}')
WHERE {id_ouro} IS NOT NULL AND {cpf_ouro} IS NOT NULL
''')

display(con.execute(f'''
SELECT
    a.origem,
    coalesce(g.n, 0) AS n,
    ROUND(100.0 * coalesce(g.n, 0) / nullif(t.n_censo, 0), 1) AS pct
FROM (VALUES
    (1, 'lista_ouro'),
    (2, 'pareamento'),
    (3, 'sem_cpf')
) a(ord, origem)
CROSS JOIN (
    SELECT COUNT(*) AS n_censo FROM {TABELA_CENSO_LIMPA}
) t
LEFT JOIN (
    SELECT
        CASE
            WHEN o.id_censo IS NOT NULL THEN 'lista_ouro'
            WHEN p.unique_id_censo IS NOT NULL THEN 'pareamento'
            ELSE 'sem_cpf'
        END AS origem,
        COUNT(*) AS n
    FROM {TABELA_CENSO_LIMPA} c
    LEFT JOIN (
        SELECT DISTINCT id_censo FROM lista_ouro
    ) o ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
    LEFT JOIN (
        SELECT DISTINCT unique_id_censo FROM atribuicao
    ) p ON p.unique_id_censo = c.unique_id
    GROUP BY 1
) g ON g.origem = a.origem
ORDER BY a.ord
''').df())

n_ambos = con.execute(f'''
SELECT COUNT(*)
FROM {TABELA_CENSO_LIMPA} c
JOIN (SELECT DISTINCT id_censo FROM lista_ouro) o
  ON o.id_censo = CAST(c.person_id_censo AS VARCHAR)
JOIN (SELECT DISTINCT unique_id_censo FROM atribuicao) p
  ON p.unique_id_censo = c.unique_id
''').fetchone()[0]
print('nos_dois:', f'{n_ambos:,}')


## Recall ouro na associação (aplicação)

Entre pares 1:1 da ouro que estão no censo de **aplicação**, quantos
associaram o CPF ouro / errado / nenhum.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE gt_ouro_app AS
SELECT
    'censo_' || o.id_censo AS unique_id_censo,
    'cpf_' || o.cpf_norm AS unique_id_cpf_ouro,
    o.id_censo,
    o.cpf_norm
FROM lista_ouro o
JOIN {TABELA_CENSO_LIMPA_APLICACAO} c
  ON CAST(c.person_id_censo AS VARCHAR) = o.id_censo
QUALIFY
    COUNT(*) OVER (PARTITION BY o.id_censo) = 1
    AND COUNT(*) OVER (PARTITION BY o.cpf_norm) = 1
''')

# Se a ouro foi excluída da aplicação, n_gt pode ser 0 — ok.
n_gt = con.execute('SELECT COUNT(*) FROM gt_ouro_app').fetchone()[0]
print('Pares ouro 1:1 no censo aplicação:', f'{n_gt:,}')

if n_gt > 0:
    display(con.execute('''
    SELECT
        CASE
            WHEN a.unique_id_cpf IS NULL THEN 'sem_associacao'
            WHEN a.unique_id_cpf = g.unique_id_cpf_ouro THEN 'associou_cpf_ouro'
            ELSE 'associou_cpf_errado'
        END AS corte,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM gt_ouro_app g
    LEFT JOIN atribuicao a ON a.unique_id_censo = g.unique_id_censo
    GROUP BY 1
    ORDER BY 1
    ''').df())
else:
    print(
        'Nenhum par ouro no censo_limpo_aplicacao '
        '(esperado se a ouro saiu no 00b). Use LISTA_OURO_AVALIACAO no 03c '
        'para validação holdout, ou confira PREFIXAR_IDS / ids dos .gz.'
    )


In [ ]:
con.close()
print('ok')
